# NumPy for Machine Learning: Vectorization Demos

This notebook demonstrates how to transition from slow, unoptimized `for` loop implementations to fast, **vectorized** operations using NumPy. We cover three common machine learning tasks with increasing difficulty.

## 1. Easy Demo: Sigmoid Activation

The sigmoid function is heavily used in logistic regression and as a neural network activation. It is defined as `1 / (1 + exp(-x))`.

In [6]:
import numpy as np
import math
import time

# Generate some dummy data (e.g., logits from a layer)
np.random.seed(42)
X_easy = np.random.randn(1000000)

def sigmoid_loop(arr):
    result = np.zeros_like(arr)
    for i in range(len(arr)):
        result[i] = 1.0 / (1.0 + math.exp(-arr[i]))
    return result

def sigmoid_vectorized(arr):
    return 1.0 / (1.0 + np.exp(-arr))

# Benchmark Loop
start = time.time()
res_loop = sigmoid_loop(X_easy)
time_loop = time.time() - start

# Benchmark Vectorized
start = time.time()
res_vec = sigmoid_vectorized(X_easy)
time_vec = time.time() - start

print(f"For-loop time: {time_loop:.5f} sec")
print(f"Vectorized time: {time_vec:.5f} sec")
print(f"Results identical: {np.allclose(res_loop, res_vec)}")

For-loop time: 0.30922 sec
Vectorized time: 0.01257 sec
Results identical: True


## 2. Medium Demo: Row-wise Cosine Similarity

Given two matrices of embeddings, $X$ and $Y$ (both shape `[N, D]`), compute the cosine similarity between each corresponding pair of rows. This is common when comparing a batch of predicted embeddings to a batch of target embeddings.

In [7]:
# Generate batches of embeddings
N, D = 10000, 128
X_med = np.random.randn(N, D)
Y_med = np.random.randn(N, D)

def cosine_sim_loop(X, Y):
    n_rows, d_cols = X.shape
    result = np.zeros(n_rows)
    for i in range(n_rows):
        # Dot product of row i
        dot_prod = sum(X[i, j] * Y[i, j] for j in range(d_cols))
        # Norm of row i in X
        norm_x = math.sqrt(sum(X[i, j]**2 for j in range(d_cols)))
        # Norm of row i in Y
        norm_y = math.sqrt(sum(Y[i, j]**2 for j in range(d_cols)))
        result[i] = dot_prod / (norm_x * norm_y)
    return result

def cosine_sim_vectorized(X, Y):
    # Row-wise dot product
    dot_prod = np.sum(X * Y, axis=1)
    # Row-wise norms
    norm_x = np.linalg.norm(X, axis=1)
    norm_y = np.linalg.norm(Y, axis=1)
    return dot_prod / (norm_x * norm_y)

# Benchmark Loop (we use a smaller subset for loop to save time, it's very slow in raw python)
subset_size = 500
start = time.time()
res_loop = cosine_sim_loop(X_med[:subset_size], Y_med[:subset_size])
time_loop = time.time() - start

# Benchmark Vectorized (on the same subset for fair verification)
start = time.time()
res_vec = cosine_sim_vectorized(X_med[:subset_size], Y_med[:subset_size])
time_vec = time.time() - start

print(f"For-loop time (subset): {time_loop:.5f} sec")
print(f"Vectorized time (subset): {time_vec:.5f} sec")
print(f"Results identical: {np.allclose(res_loop, res_vec)}")

For-loop time (subset): 0.07984 sec
Vectorized time (subset): 0.00000 sec
Results identical: True


## 3. Hard Demo: Pairwise Distance Matrix (Broadcasting)

Given a dataset $X$ of shape `[N, D]` and queries $Y$ of shape `[M, D]`, output an `[N, M]` matrix where the $(i, j)$ element is the Euclidean distance between $X_i$ and $Y_j$. This is the core operation behind algorithms like K-Nearest Neighbors and clustering.

In [8]:
N_pts, M_pts, D_dim = 1000, 500, 64
X_hard = np.random.randn(N_pts, D_dim)
Y_hard = np.random.randn(M_pts, D_dim)

def pairwise_dist_loop(X, Y):
    n, d = X.shape
    m, _ = Y.shape
    dist_matrix = np.zeros((n, m))
    for i in range(n):
        for j in range(m):
            sq_dist = 0.0
            for k in range(d):
                sq_dist += (X[i, k] - Y[j, k]) ** 2
            dist_matrix[i, j] = math.sqrt(sq_dist)
    return dist_matrix

def pairwise_dist_vectorized(X, Y):
    # Using broadcasting: X[:, np.newaxis, :] expands to (N, 1, D)
    # Y[np.newaxis, :, :] expands to (1, M, D)
    # Difference is (N, M, D). Sum squared diffs over axis 2.
    diff = X[:, np.newaxis, :] - Y[np.newaxis, :, :]
    return np.sqrt(np.sum(diff ** 2, axis=-1))

    # Alternative memory-efficient approach utilizing (x-y)^2 = x^2 + y^2 - 2xy:
    # x2 = np.sum(X**2, axis=1, keepdims=True)
    # y2 = np.sum(Y**2, axis=1)
    # xy = np.dot(X, Y.T)
    # return np.sqrt(x2 + y2 - 2*xy)

subset_N, subset_M = 100, 100
start = time.time()
res_loop = pairwise_dist_loop(X_hard[:subset_N], Y_hard[:subset_M])
time_loop = time.time() - start

start = time.time()
res_vec = pairwise_dist_vectorized(X_hard[:subset_N], Y_hard[:subset_M])
time_vec = time.time() - start

print(f"For-loop time (subset): {time_loop:.5f} sec")
print(f"Vectorized time (subset): {time_vec:.5f} sec")
print(f"Results identical: {np.allclose(res_loop, res_vec)}")

For-loop time (subset): 0.25499 sec
Vectorized time (subset): 0.00566 sec
Results identical: True
